In [1]:
1+2

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 059a7aac-377d-43d3-8b90-f0b203994c3e
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 059a7aac-377d-43d3-8b90-f0b203994c3e to get into ready status...
Session 059a7aac-377d-43d3-8b90-f0b203994c3e has been created.
3


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName('read gold data').getOrCreate()
spark

In [3]:
role_demand_path = 's3://job-market-tracker-hsg/gold/role_demand/'
role_location_demand_path = 's3://job-market-tracker-hsg/gold/role_location_demand/'
role_trend_path = 's3://job-market-tracker-hsg/gold/role_trend/'
top_skills_by_role_path = 's3://job-market-tracker-hsg/gold/top_skills_by_role/'

In [4]:
role_demand_df = spark.read.parquet(role_demand_path)
role_location_demand_df = spark.read.parquet(role_location_demand_path)
role_trend_df = spark.read.parquet(role_trend_path)
top_skills_by_role_df = spark.read.parquet(top_skills_by_role_path)

In [5]:
role_demand_df.show()

+-----------------+---------+
|standardized_role|job_count|
+-----------------+---------+
|    Data Engineer|        6|
|     Data Analyst|        2|
|   Data Scientist|        2|
+-----------------+---------+


In [6]:
role_location_demand_df.show()

+-----------------+-------------+---------+
|standardized_role|location_city|job_count|
+-----------------+-------------+---------+
|     Data Analyst|      Chennai|        1|
|    Data Engineer|    Bengaluru|        3|
|   Data Scientist|    Bengaluru|        1|
|    Data Engineer|       Mumbai|        2|
|    Data Engineer|    Hyderabad|        1|
|     Data Analyst|         null|        1|
|   Data Scientist|      Chennai|        1|
+-----------------+-------------+---------+


In [7]:
role_trend_df.show()

+-----------------+----+---------+
|standardized_role|year|job_count|
+-----------------+----+---------+
|    Data Engineer|2025|        6|
|     Data Analyst|2025|        2|
|   Data Scientist|2025|        2|
+-----------------+----+---------+


In [15]:
top_skills_by_role_df.filter(col('standardized_role')=='Data Engineer').orderBy('standardized_role', desc('demand_count')).show()

+-----------------+------------+------------+
|standardized_role|       skill|demand_count|
+-----------------+------------+------------+
|    Data Engineer|        Java|           5|
|    Data Engineer|         SQL|           4|
|    Data Engineer|      Python|           4|
|    Data Engineer|         AWS|           3|
|    Data Engineer|       Azure|           3|
|    Data Engineer|       Kafka|           2|
|    Data Engineer|       Spark|           2|
|    Data Engineer|       Flink|           2|
|    Data Engineer|       NoSQL|           1|
|    Data Engineer|    DynamoDB|           1|
|    Data Engineer|         GCP|           1|
|    Data Engineer|Apache Spark|           1|
|    Data Engineer|       Scala|           1|
|    Data Engineer|        Akka|           1|
|    Data Engineer|Google Cloud|           1|
|    Data Engineer| Spring Boot|           1|
|    Data Engineer|         dbt|           1|
|    Data Engineer|      Hadoop|           1|
|    Data Engineer|  API design|  

In [13]:
top_skills_by_role_df.groupBy('standardized_role').agg(
    collect_set('skill').alias('unique_skills')
).withColumn(
    'skills_count',
    size('unique_skills')
).show()

+-----------------+--------------------+------------+
|standardized_role|       unique_skills|skills_count|
+-----------------+--------------------+------------+
|    Data Engineer|[SQL, AWS, Scala,...|          26|
|     Data Analyst|[Oracle, PostgreS...|           8|
|   Data Scientist|[AWS, Docker, GCP...|          14|
+-----------------+--------------------+------------+
